In [4]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import os

DB_USER     = "postgres"
DB_PASSWORD = quote_plus("DJR@work17")  # encodes @ safely
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "consumer_behaviour_db"

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

try:
    engine = create_engine(connection_string)
    with engine.connect() as conn:
        result  = conn.execute(text("SELECT version()"))
        version = result.fetchone()[0]
    print("✅ Connected to PostgreSQL successfully")
    print(f"   {version[:60]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to PostgreSQL successfully
   PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35


In [5]:
# ============================================================
# CREATE TABLES AND LOAD ALL CLEANED DATA
# ============================================================
# WHY WE DEFINE SCHEMAS:
# A schema tells PostgreSQL exactly what type of data
# each column holds. This enforces data integrity —
# you cannot accidentally store text in a number column.
# It also allows PostgreSQL to optimise storage and queries.
# ============================================================

# Load all cleaned datasets
print("Loading cleaned datasets...")
instagram_df  = pd.read_csv("data/cleaned/instagram_clean.csv")
trends_df     = pd.read_csv("data/cleaned/trends_clean.csv")
retail_df     = pd.read_csv("data/cleaned/retail_clean.csv")
consumers_df  = pd.read_csv("data/cleaned/chennai_consumers.csv")
stores_df     = pd.read_csv("data/cleaned/chennai_stores.csv")
monthly_df    = pd.read_csv("data/cleaned/monthly_trends.csv")
posts_df      = pd.read_csv("data/cleaned/chennai_posts_with_sentiment.csv")

print("✅ All datasets loaded")
print()

# Load each dataset into PostgreSQL
# if_exists='replace' → drops and recreates table each run
# if_exists='append'  → adds rows to existing table
# index=False         → don't write the pandas row numbers
datasets = {
    "instagram_posts"     : instagram_df,
    "social_media_trends" : trends_df,
    "retail_transactions" : retail_df,
    "chennai_consumers"   : consumers_df,
    "chennai_stores"      : stores_df,
    "monthly_trends"      : monthly_df,
    "sentiment_posts"     : posts_df,
}

print("Loading data into PostgreSQL tables...")
print()
for table_name, df in datasets.items():
    try:
        df.to_sql(
            table_name,
            engine,
            if_exists = "replace",
            index     = False,
            chunksize = 1000   # load 1000 rows at a time
        )
        print(f"   ✅ {table_name:30s} {len(df):>6} rows loaded")
    except Exception as e:
        print(f"   ❌ {table_name:30s} Failed: {e}")

print()
print("✅ All tables created in consumer_behaviour_db")

Loading cleaned datasets...
✅ All datasets loaded

Loading data into PostgreSQL tables...

   ✅ instagram_posts                 29999 rows loaded
   ✅ social_media_trends              5000 rows loaded
   ✅ retail_transactions              1000 rows loaded
   ✅ chennai_consumers                5000 rows loaded
   ✅ chennai_stores                    200 rows loaded
   ✅ monthly_trends                    216 rows loaded
   ✅ sentiment_posts                  3000 rows loaded

✅ All tables created in consumer_behaviour_db


In [6]:
# ============================================================
# VERIFY ALL TABLES ARE IN THE DATABASE
# ============================================================

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name, 
               pg_size_pretty(
                   pg_total_relation_size(
                       quote_ident(table_name)
                   )
               ) as size
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
    """))
    tables = result.fetchall()

print("Tables in consumer_behaviour_db:")
print("-" * 40)
for table in tables:
    print(f"   {table[0]:30s} {table[1]}")

Tables in consumer_behaviour_db:
----------------------------------------
   chennai_consumers              872 kB
   chennai_stores                 64 kB
   instagram_posts                7776 kB
   monthly_trends                 64 kB
   retail_transactions            192 kB
   sentiment_posts                720 kB
   social_media_trends            704 kB


In [8]:
queries = {}

# ── Query 1: Platform popularity by age group ────────────────
queries["Q1_platform_by_age"] = """
SELECT 
    age_group,
    primary_platform,
    COUNT(*) as consumer_count,
    ROUND(AVG(monthly_spend_inr)::numeric, 0) as avg_spend,
    ROUND(AVG(daily_screen_time_hrs)::numeric, 1) as avg_screen_time
FROM chennai_consumers
GROUP BY age_group, primary_platform
ORDER BY age_group, consumer_count DESC;
"""

# ── Query 2: Top purchase category by content genre ──────────
queries["Q2_purchase_by_genre"] = """
SELECT 
    top_content_genre,
    top_purchase_category,
    COUNT(*) as consumer_count,
    ROUND(AVG(monthly_spend_inr)::numeric, 0) as avg_monthly_spend,
    ROUND(AVG(daily_screen_time_hrs)::numeric, 2) as avg_screen_time
FROM chennai_consumers
GROUP BY top_content_genre, top_purchase_category
ORDER BY top_content_genre, consumer_count DESC;
"""

# ── Query 3: Locality commercial performance ─────────────────
queries["Q3_locality_performance"] = """
SELECT 
    locality,
    COUNT(*) as store_count,
    ROUND(AVG(avg_monthly_sales_inr)::numeric, 0) as avg_sales,
    ROUND(AVG(avg_monthly_profit_inr)::numeric, 0) as avg_profit,
    ROUND(AVG(footfall_per_day)::numeric, 0) as avg_footfall,
    ROUND(AVG(rating)::numeric, 2) as avg_rating,
    SUM(CASE WHEN has_online_presence THEN 1 ELSE 0 END) 
        as online_stores
FROM chennai_stores
GROUP BY locality
ORDER BY avg_sales DESC;
"""

# ── Query 4: Sentiment analysis summary ──────────────────────
queries["Q4_sentiment_summary"] = """
SELECT 
    content_genre,
    COUNT(*) as total_posts,
    ROUND(AVG(sent_compound)::numeric, 3) as avg_sentiment,
    SUM(CASE WHEN sent_label = 'positive' THEN 1 ELSE 0 END) 
        as positive_count,
    SUM(CASE WHEN sent_label = 'negative' THEN 1 ELSE 0 END) 
        as negative_count,
    SUM(CASE WHEN sent_label = 'neutral'  THEN 1 ELSE 0 END) 
        as neutral_count,
    ROUND(AVG(views)::numeric, 0) as avg_views,
    ROUND(AVG(likes)::numeric, 0) as avg_likes
FROM sentiment_posts
GROUP BY content_genre
ORDER BY avg_sentiment DESC;
"""

# ── Query 5: Screen time vs impulse buying ───────────────────
queries["Q5_screentime_impulse"] = """
SELECT 
    impulse_buy_freq,
    COUNT(*) as consumer_count,
    ROUND(AVG(daily_screen_time_hrs)::numeric, 2) as avg_screen_time,
    ROUND(AVG(monthly_spend_inr)::numeric, 0) as avg_spend
FROM chennai_consumers
GROUP BY impulse_buy_freq
ORDER BY avg_screen_time DESC;
"""

# ── Query 6: Monthly revenue trend by category ───────────────
queries["Q6_monthly_revenue"] = """
SELECT 
    month,
    category,
    total_revenue,
    festival_boost,
    is_trending,
    ROUND(
        (total_revenue - LAG(total_revenue) 
            OVER (PARTITION BY category ORDER BY month_date)
        )::numeric * 100.0 / NULLIF(
            LAG(total_revenue) 
            OVER (PARTITION BY category ORDER BY month_date), 0
        ), 1
    ) as month_over_month_pct
FROM monthly_trends
WHERE record_type = 'retail_sales'
ORDER BY category, month_date;
"""

# ── Query 7: Top performing stores ───────────────────────────
queries["Q7_top_stores"] = """
SELECT 
    store_name,
    store_category,
    store_scale,
    locality,
    avg_monthly_sales_inr,
    avg_monthly_profit_inr,
    footfall_per_day,
    rating,
    has_online_presence,
    years_in_business
FROM chennai_stores
WHERE avg_monthly_sales_inr > (
    SELECT AVG(avg_monthly_sales_inr) 
    FROM chennai_stores
)
ORDER BY avg_monthly_sales_inr DESC
LIMIT 20;
"""

# ── Query 8: Consumer segmentation ───────────────────────────
queries["Q8_consumer_segments"] = """
SELECT 
    age_group,
    income_bracket,
    COUNT(*) as segment_size,
    ROUND(AVG(monthly_spend_inr)::numeric, 0) as avg_spend,
    ROUND(AVG(daily_screen_time_hrs)::numeric, 1) as avg_screen_time,
    MODE() WITHIN GROUP (ORDER BY primary_platform) 
        as dominant_platform,
    MODE() WITHIN GROUP (ORDER BY top_content_genre) 
        as dominant_genre,
    MODE() WITHIN GROUP (ORDER BY top_purchase_category) 
        as dominant_purchase
FROM chennai_consumers
GROUP BY age_group, income_bracket
ORDER BY age_group, avg_spend DESC;
"""

# ── Run all queries ───────────────────────────────────────────
print("=" * 55)
print("RUNNING ALL SQL QUERIES")
print("=" * 55)

results = {}
for query_name, sql in queries.items():
    try:
        with engine.connect() as conn:
            df_result = pd.read_sql(text(sql), conn)
        results[query_name] = df_result
        print(f"\n✅ {query_name}")
        print(df_result.head(3).to_string(index=False))
        print(f"   ({len(df_result)} rows returned)")
    except Exception as e:
        print(f"\n❌ {query_name} — Error: {e}")

RUNNING ALL SQL QUERIES

✅ Q1_platform_by_age
age_group primary_platform  consumer_count  avg_spend  avg_screen_time
    13-17        instagram             223    21984.0              5.1
    13-17           tiktok             190    16195.0              5.2
    13-17          youtube             100    15762.0              5.3
   (20 rows returned)

✅ Q2_purchase_by_genre
top_content_genre top_purchase_category  consumer_count  avg_monthly_spend  avg_screen_time
        education                 books             366            18198.0             3.27
        education           electronics             218            14463.0             3.11
        education              clothing             150            17519.0             3.24
   (24 rows returned)

✅ Q3_locality_performance
 locality  store_count  avg_sales  avg_profit  avg_footfall  avg_rating  online_stores
    Adyar           15  1575198.0    275423.0         592.0        4.16             12
      OMR           19  1320422.0